In [9]:
import sys
sys.path.append("../src")

from pyspark.sql import SparkSession, functions as F
import main, m1

spark = (SparkSession.builder
         .master("local[*]")
         .appName("dev-m2")
         .config("spark.driver.memory", "2g")
         .getOrCreate())

trips = main.load_trips(spark, "../databases", [2025], 1.0)        
clean = m1.clean(m1.flag_invalid(trips))                           
zones = main.load_zones(spark, "../databases/taxi_zone_lookup.csv")   

print(clean.count())
zones.show(5)   

2791183
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [10]:
pickup_zone = zones.select(F.col("LocationID").alias("PULocationID"),   
                           F.col("Borough").alias("pickup_borough"))      

with_borough = clean.join(pickup_zone, "PULocationID")
with_borough.select("tpep_pickup_datetime", "PULocationID", "pickup_borough").show(5)
print(with_borough.count())

+--------------------+------------+--------------+
|tpep_pickup_datetime|PULocationID|pickup_borough|
+--------------------+------------+--------------+
| 2025-01-01 00:18:38|         229|     Manhattan|
| 2025-01-01 00:32:40|         236|     Manhattan|
| 2025-01-01 00:44:04|         141|     Manhattan|
| 2025-01-01 00:14:27|         244|     Manhattan|
| 2025-01-01 00:21:34|         244|     Manhattan|
+--------------------+------------+--------------+
only showing top 5 rows
2791183


In [11]:
with_cols = (with_borough
    .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")) 
    .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))     
    .withColumn("duration_min",                                       
                (F.unix_timestamp("tpep_dropoff_datetime") -
                 F.unix_timestamp("tpep_pickup_datetime")) / 60.0))

with_cols.select("tpep_pickup_datetime", "tpep_dropoff_datetime",
                 "day_of_week", "hour_of_day", "duration_min").show(5)

+--------------------+---------------------+-----------+-----------+-----------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|day_of_week|hour_of_day|     duration_min|
+--------------------+---------------------+-----------+-----------+-----------------+
| 2025-01-01 00:18:38|  2025-01-01 00:26:59|          4|          0|             8.35|
| 2025-01-01 00:32:40|  2025-01-01 00:35:13|          4|          0|             2.55|
| 2025-01-01 00:44:04|  2025-01-01 00:46:01|          4|          0|             1.95|
| 2025-01-01 00:14:27|  2025-01-01 00:20:01|          4|          0|5.566666666666666|
| 2025-01-01 00:21:34|  2025-01-01 00:25:06|          4|          0|3.533333333333333|
+--------------------+---------------------+-----------+-----------+-----------------+
only showing top 5 rows


In [12]:
demand = (with_cols
    .groupBy("pickup_borough", "day_of_week", "hour_of_day")    
    .agg(F.count("*").alias("trips"),                          
         F.avg("fare_amount").alias("avg_fare"),               
         F.avg("duration_min").alias("avg_duration_min"),      
         F.avg("trip_distance").alias("avg_distance_mi")))   

demand.filter("pickup_borough = 'Manhattan' AND day_of_week = 2").orderBy("hour_of_day").show(24)
demand.agg(F.sum("trips")).show()

+--------------+-----------+-----------+-----+------------------+------------------+------------------+
|pickup_borough|day_of_week|hour_of_day|trips|          avg_fare|  avg_duration_min|   avg_distance_mi|
+--------------+-----------+-----------+-----+------------------+------------------+------------------+
|     Manhattan|          2|          0| 3076|14.192649544863462| 11.23166449934979|2.6221261378413536|
|     Manhattan|          2|          1| 1559|13.964650416933925| 9.736027367970918| 2.568422065426555|
|     Manhattan|          2|          2|  876|15.235605022831045|10.666552511415526| 2.904235159817351|
|     Manhattan|          2|          3|  678| 19.10530973451327|13.217305801376607|3.8713864306784664|
|     Manhattan|          2|          4|  735| 26.76738775510204|14.348775510204085| 5.815183673469387|
|     Manhattan|          2|          5| 1363| 19.67391049156273| 11.56972364881389| 4.086228906823185|
|     Manhattan|          2|          6| 3937|15.021595123190252

[Stage 69:====================================>                     (5 + 3) / 8]

+----------+
|sum(trips)|
+----------+
|   2791183|
+----------+



In [13]:
demand.write.mode("overwrite").partitionBy("pickup_borough").parquet("../output/m2_demand")

import os
print(sorted(os.listdir("../output/m2_demand")))

['._SUCCESS.crc', '_SUCCESS', 'pickup_borough=Bronx', 'pickup_borough=Brooklyn', 'pickup_borough=EWR', 'pickup_borough=Manhattan', 'pickup_borough=Queens', 'pickup_borough=Staten Island']


In [14]:
import m2

demand2 = m2.build(clean, zones)
m2.write(demand2, "../output")
demand2.agg(F.sum("trips")).show()   

[Stage 84:===========================================>              (6 + 2) / 8]

+----------+
|sum(trips)|
+----------+
|   2791183|
+----------+

